# FieldSense: your own field, anywhere in Canada

Enter a location. FieldSense finds **the field at that point** on Canada's crop map and shows:

1. **Its history:** the crop each year (AAFC crop map) and every clear NASA satellite view since 2021, as a barcode (brown = bare soil, green = growing).
2. **Its soil cover, measured:** how much of the season it was bare, and how much it was green outside summer. Plus a check of FieldSense's crop reading against the crop map.
3. **Next season (2027) weather:** NASA POWER corrected with the nearest Environment Canada station, and 1,000 possible seasons.
4. **Which rotation, and why** (for corn–soybean–wheat farms; other systems are the next step).

**You need:** a free NASA Earthdata account ([register here](https://urs.earthdata.nasa.gov/users/new)). **Time:** about 15–20 minutes, mostly reading the satellite images.

**Privacy:** everything runs in your own Colab session. Your field's outline and results are not sent anywhere or published.

Click **Runtime → Run all**, or run the cells one by one.


In [ ]:
# 1. Install FieldSense (about 1 minute)
!pip install -q "fieldsense[hls] @ git+https://github.com/SetarehAlamdar/FieldSense.git"

## 2. Your location

Paste latitude, longitude **from Google Maps**: right-click on your field → click the numbers at the top of the menu to copy them. Put the point in the middle of the field.


In [ ]:
LOCATION = "43.6412, -80.4180"  #@param {type:"string"}
NAME = "My field"  #@param {type:"string"}

LAT, LON = [float(v) for v in LOCATION.replace(";", ",").split(",")[:2]]
assert 41.5 < LAT < 60 and -141 < LON < -52, "That doesn't look like a location in southern Canada (latitude, longitude)."
print(f"{NAME}: latitude {LAT:.4f}, longitude {LON:.4f}")

## 3. Log in to NASA Earthdata

Type your Earthdata username and password when asked (they stay in this session). If you use Colab Secrets, add `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD` there instead.


In [ ]:
import os, json
import earthaccess
from fieldsense.sources import hls
try:
    from google.colab import userdata
    os.environ["EARTHDATA_USERNAME"] = userdata.get("EARTHDATA_USERNAME")
    os.environ["EARTHDATA_PASSWORD"] = userdata.get("EARTHDATA_PASSWORD")
    earthaccess.login(strategy="environment")
except Exception:
    earthaccess.login(strategy="interactive")
hls.use_earthaccess_login()
CACHE = "/content/fieldsense_cache"
os.makedirs(CACHE, exist_ok=True)

## 4. Find your field on the crop map


In [ ]:
from fieldsense import myfield as mf
FIELD = mf.field_at_point(LON, LAT, CACHE)
if FIELD is None:
    raise SystemExit("No farm field found at this point: move the point to the middle of a field and run again.")
if not FIELD.get("farmland", True):
    raise SystemExit(f"This point is {FIELD['land']} on Canada's crop map, not a farm field. "
                     "In Google Maps (Satellite view), right-click in the middle of a field and use that location.")
print(f"Field at your point: {FIELD['area_ha']} ha, rotation type: {FIELD['group'] or 'not simple/diverse (e.g. permanent grass)'}")
for y in sorted(FIELD["crops"]):
    print(f"  {y}: {FIELD['crops'][y]:15s} ({FIELD['purity'][y]:.0%} of the field's pixels agree)")

## 5. Read every clear satellite view (10–15 minutes)

Every NASA HLS image (Landsat 8/9 and Sentinel-2, 30 m) of your field, 2021–2025, with clouds and snow removed. Progress is printed every 50 images.


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from fieldsense import canada as ca
from fieldsense.phenology import field_history

ndvi_file = os.path.join(CACHE, "my_field_ndvi.csv")
ca.hls_ndvi_for_fields({1: FIELD["lonlat"]}, ndvi_file)
ndvi = pd.read_csv(ndvi_file, parse_dates=["date"])
f = {"id": 1, "name": NAME, "group": FIELD["group"] or "other", "codes": FIELD["codes"]}
REC = ca.region_records(ndvi, [f])[0]
obs = ndvi[ndvi["ndvi"].notna()][["date", "ndvi"]]
HIST = field_history(obs).to_dict("records")

bc = REC["barcode"]
vals = np.array([[np.nan if v is None else v / 100 for v in r] for r in bc["values"]])
cmap = LinearSegmentedColormap.from_list("g", [(0, "#8c5a2b"), (0.25, "#d9c48c"), (0.5, "#c9dd8a"), (0.75, "#5aa040"), (1, "#1f6b2a")])
cmap.set_bad("#e2e4e8")
fig, ax = plt.subplots(figsize=(10, 0.5 + 0.45 * len(bc["years"])))
ax.imshow(np.ma.masked_invalid(vals), aspect="auto", cmap=cmap, vmin=0.1, vmax=0.9, interpolation="nearest")
ax.set_yticks(range(len(bc["years"]))); ax.set_yticklabels([f"{y}  {FIELD['crops'].get(y, '')}" for y in bc["years"]])
ax.set_xticks([0, 59, 120, 181, 243, 304]); ax.set_xticklabels(["Jan", "Mar", "May", "Jul", "Sep", "Nov"])
ax.set_title(f"{NAME}: greenness every day (brown = bare soil, green = growing, grey = no clear view)", fontsize=10)
plt.tight_layout(); plt.show()

c = REC["cover"]
print(f"Bare soil: {c['bare_share_season']:.0%} of the season (1 Mar-30 Nov) | bare in spring storm season: "
      f"{c['spring_bare_share']:.0%} | green outside summer: {c['green_share_off_season']:.0%}")
ok = sum(x["agree"] for x in REC["check"])
print(f"Crop reading check: the satellite's crop timing agrees with the crop map in {ok} of {len(REC['check'])} years")
for x in REC["check"]:
    print(f"  {x['year']}: map says {x['map']} ({x['map_class']}), satellite says {x['satellite_class']}" + ("" if x["agree"] else "   <- differs"))

**How to read it:** each row is one year. Long brown stretches in spring and fall are bare soil, which is when heavy rain washes topsoil away. Green outside summer means winter wheat, hay or a cover crop protecting the soil.

## 6. Next season (2027) weather at your field


In [ ]:
from fieldsense import anywhere as aw
CLIM = aw.climate_for_point(LAT, LON, cache_dir=CACHE)
s, st = CLIM["summary"], CLIM["station"]
print("Station:", f"{st['name']} (ECCC {st['id']}), {st['km']} km away" if st else "none usable nearby: NASA POWER uncorrected")
cor = CLIM["correction"]
if cor.get("rain"):
    print(f"Rain correction, error on held-out years: {cor['rain']['held_out_error_raw_pct']}% -> {cor['rain']['held_out_error_corrected_pct']}%")
if cor.get("temperature"):
    print(f"Night temperature bias: {cor['temperature']['bias_raw_c']} -> {cor['temperature']['bias_corrected_c']} C")
print(f"Typical year: {s['annual_precip_mm']:.0f} mm rain, {s['gdd10_may_sep']:.0f} growing degree-days, "
      f"{s['frost_free_days']:.0f} frost-free days. Warming {CLIM['trend']['c_per_decade']} C/decade.")
print(f"\nOut of 1,000 possible 2027 seasons:")
for k, v in sorted(CLIM["odds"]["risks"].items(), key=lambda kv: -kv[1]["share"]):
    print(f"  {round(100*v['share']):3d} in 100   {v['text']}   ({v['past_years']} past years)")

## 7. Which rotation, and why


In [ ]:
measured = {"green_share_off_season": REC["cover"]["green_share_off_season"],
            "spring_bare_share": REC["cover"]["spring_bare_share"]}
ADV = mf.rotation_advice(HIST, CLIM["odds"], measured, FIELD["codes"])
if not ADV["available"]:
    print(ADV["reason"])
else:
    print(f"Your field now: {ADV['current_name']} (read from the satellite)")
    if ADV["keep"]:
        print(f"Suggestion: keep it. It comes out best under {ADV['robustness'][ADV['current_name']]:.0%} of 2,000 different priority mixes.")
    else:
        print(f"Suggestion: {ADV['suggested_name']}. It comes out best under {ADV['robustness'][ADV['suggested_name']]:.0%} "
              f"of 2,000 different priority mixes (your current rotation: {ADV['robustness'].get(ADV['current_name'], 0):.0%}).")
    print(f"If you keep your rotation, 2027 is: {ADV['next_crop_if_kept']}")
    for k, parts in ADV["why"].items():
        print(f"\nWhy ({'now' if k == ADV['current'] else 'suggested'}):")
        for part in parts.values():
            print(f"  - {part['text']}  [{part['evidence']}]")
    show = ["green_share_off_season", "spring_bare_share", "climate_risk_total", "legume_share", "crops_to_manage"]
    display(ADV["table"][show].round(2))

**What the evidence labels mean:** *measured* comes straight from satellite or station data; *validated* was checked against independent records (the Elora trial, Canada's crop map); *modelled* comes from a model; *assumed* is a rule of thumb.

**Limits:** no yield, profit or labour yet. Rotation options and risk thresholds are for corn–soybean–wheat systems (Ontario, Quebec). The crop map is about 85% accurate and doesn't record cover crops.

FieldSense is open source: [github.com/SetarehAlamdar/FieldSense](https://github.com/SetarehAlamdar/FieldSense) · NASA Space Apps 2026, Team Star Farmer.
